# 2.1 Convergence study

Two Abinit parameters control the accuracy of a planewave DFT calculation
for a given pseudopotential set: the plane-wave cutoff `ecut` and the
density of the k-point mesh. Both flows below were already run for you
(`flow_gaas_convecut/`, `flow_gaas_convkpt/`, next to this notebook) --
built with `../Examples/make_gaas_convecut.py` and
`../Examples/make_gaas_convkpt.py` respectively.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from abipy import abilab
abilab.enable_notebook()

%matplotlib inline

import workshop_lib as wlib

## ecut convergence

In [ ]:
print(open("../Examples/make_gaas_convecut.py").read())

In [ ]:
with abilab.GsrRobot.from_dir("flow_gaas_convecut") as robot:
    table = robot.get_dataframe()

table[["ecut", "energy", "pressure"]].sort_values("ecut")

`abipy.tools.plotting.ConvergenceAnalyzer` fits/plots a quantity against a
convergence parameter and reports the smallest parameter value for which
the target quantity stays within a given tolerance:

In [ ]:
from abipy.tools.plotting import ConvergenceAnalyzer

ecut_Ha = table.sort_values("ecut")["ecut"].tolist()
ene_per_atom_eV = (table.sort_values("ecut")["energy"] / len(wlib.gaas_structure())).tolist()

ca = ConvergenceAnalyzer.from_xy_label_vals(
    "ecut (Ha)", ecut_Ha, "E/natom (eV)", ene_per_atom_eV, tols=1e-3)
ca.plot();

## k-point convergence

Same idea, but now `ecut` is fixed and we vary the k-mesh via
`set_autokmesh(nk)`, which generates an increasingly dense homogeneous mesh.

In [ ]:
print(open("../Examples/make_gaas_convkpt.py").read())

In [ ]:
k_recip_dist, ene_per_atom_eV = [], []

with abilab.GsrRobot.from_dir("flow_gaas_convkpt") as robot:
    for label, gsr in robot:
        ene_per_atom_eV.append(gsr.energy_per_atom)
        rprim = gsr.structure.lattice.matrix
        kptrlatt = gsr.kpoints.ksampling["kptrlatt"]
        R_latt = np.dot(kptrlatt, rprim)
        k_latt = 2 * np.pi * np.linalg.inv(R_latt)
        kmin = max(np.linalg.norm(k) for k in k_latt)
        k_recip_dist.append(1 / kmin)

ca = ConvergenceAnalyzer.from_xy_label_vals(
    "Inverse k-point distance (Ang)", k_recip_dist,
    "E/natom (eV)", ene_per_atom_eV, tols=1e-3)
ca.plot();

> **Exercise.** Redo the ecut convergence study for silicon
> (`wlib.si_structure()`), reusing `wlib.gs_input` as a template: build a
> flow with one SCF task per `ecut`, run it, then analyze it here with a
> `GsrRobot` and `ConvergenceAnalyzer` as above. Is the converged `ecut`
> similar to the one you found for GaAs? Should it be? (This one's also
> waiting for you in `4-Challenge.ipynb` if you'd rather come back to it
> later.)

Continue with [`2.2-Relaxation.ipynb`](2.2-Relaxation.ipynb).